In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob # The perfect library for finding files

# --- 1. PATH SETUP ---

# The "base" path where folders S03, S08, etc., are located
BASE_DATA_DIR = '../data' 

# The output folder for global results (as you requested)
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, 'all', 'delay')

# Create the output folder if it doesn't exist (as you requested)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"All aggregated results will be saved in: {OUTPUT_DIR}")


# --- 2. FINDING AND LOADING ALL CSVs ---

# Create a search pattern to find ALL CSV files
# in ../data/S[ANYTHING]/delay/*.csv
search_pattern = os.path.join(BASE_DATA_DIR, 'S*', 'delay', '*.csv')

# glob finds all files matching the pattern
all_csv_files = glob.glob(search_pattern)

if not all_csv_files:
    print(f"ERROR: No CSV files found. Check the path: {search_pattern}")
    # Exit the script if nothing is found
    exit()

print(f"\nFound {len(all_csv_files)} CSV files to aggregate...")

# A list where we will put all the individual latency DataFrames
all_latencies_list = []

for csv_file in all_csv_files:
    # print(f"Loading: {csv_file}") # Uncomment for debug
    try:
        df = pd.read_csv(csv_file, sep=';', decimal='.')
        
        # --- IMPORTANT: Cleaning ---
        # Remove the "Mean_Latency" row that we might have added
        # We look for rows where 'Latency_ms' is numeric, coercing errors
        df_clean = df[pd.to_numeric(df['Latency_ms'], errors='coerce').notna()]
        
        # Convert the column to numbers
        latencies_series = pd.to_numeric(df_clean['Latency_ms'])
        
        all_latencies_list.append(latencies_series)
        
    except Exception as e:
        print(f"Error while reading {csv_file}: {e}")

# --- 3. CREATING THE GLOBAL DATAFRAME ---

if not all_latencies_list:
    print("ERROR: No valid latency data was loaded. Aborting.")
    exit()

# Concatenate all individual Series into one large Series
final_latency_series = pd.concat(all_latencies_list, ignore_index=True)

print(f"\n--- Aggregation complete. ---")
print(f"Total number of latencies calculated (from all subjects): {len(final_latency_series)}")


# --- 4. CALCULATING KEY METRICS (FOR YOUR THESIS) ---

# Calculate the statistics you asked for
mean_lat = final_latency_series.mean()
std_lat = final_latency_series.std()
median_lat = final_latency_series.median()
min_lat = final_latency_series.min()
max_lat = final_latency_series.max()
count_lat = len(final_latency_series)

# Adding the 95th Percentile (P95) - a VERY strong metric for the thesis
# Answers the question: "In 95% of cases, the latency was LOWER than...?"
p95_lat = final_latency_series.quantile(0.95)

print("\n--- GLOBAL LATENCY STATISTICS (All Subjects) ---")
print(f"Mean Latency: {mean_lat:.2f} ms")
print(f"Std Deviation: {std_lat:.2f} ms")
print(f"Median (P50): {median_lat:.2f} ms")
print(f"P95 (95th Percentile): {p95_lat:.2f} ms")
print(f"Min: {min_lat:.2f} ms | Max: {max_lat:.2f} ms")


# --- 5. SAVING METRICS TO FILE ---

metrics_file_path = os.path.join(OUTPUT_DIR, 'summary_global_metrics.txt')

with open(metrics_file_path, 'w', encoding='utf-8') as f:
    f.write("--- Global Latency Metrics Summary (T_myo - T_real) ---\n")
    f.write(f"Subjects Analyzed: {len(all_csv_files)}\n")
    f.write(f"Total Latencies Calculated: {count_lat}\n")
    f.write("----------------------------------------------------------\n")
    f.write(f"Mean: {mean_lat:.2f} ms\n")
    f.write(f"Median (P50): {median_lat:.2f} ms\n")
    f.write(f"Standard Deviation: {std_lat:.2f} ms\n")
    f.write(f"P95 (95th Percentile): {p95_lat:.2f} ms\n")
    f.write(f"Minimum: {min_lat:.2f} ms\n")
    f.write(f"Maximum: {max_lat:.2f} ms\n")

print(f"\nMetrics summary saved to: {metrics_file_path}")


# --- 6. CREATING AND SAVING PLOTS (WITH CUSTOM FONT SIZES) ---

# --- Font Size Definitions (from previous script) ---
LABEL_SIZE = 22
TICK_SIZE = 20
LEGEND_SIZE = 20 # Matching tick size for consistency

# --- PLOT 1: The Histogram (The most important one) ---

hist_file_path = os.path.join(OUTPUT_DIR, 'global_latency_histogram.png')

# Create figure and axes object
fig, ax = plt.subplots(figsize=(10, 6))

# Plot histogram on the axes
final_latency_series.hist(bins=100, ax=ax) 

# --- STYLING ---
# ax.set_title(...) # <-- REMOVED
# ***** MODIFICATO *****
ax.set_xlabel('System Correction Latency', fontsize=LABEL_SIZE)
ax.set_ylabel('Occurrences', fontsize=LABEL_SIZE)

# ***** AGGIUNTO *****
# Imposta il limite massimo dell'asse X
ax.set_xlim(left=0, right=1500)

# Apply tick sizes
ax.tick_params(axis='x', labelsize=TICK_SIZE)
ax.tick_params(axis='y', labelsize=TICK_SIZE)

# Add vertical lines for Mean and Median
mean_label = f'Mean ({mean_lat:.2f} ms)'
median_label = f'Median ({median_lat:.2f} ms)'
ax.axvline(mean_lat, color='red', linestyle='dashed', linewidth=2, label=mean_label)
ax.axvline(median_lat, color='green', linestyle='dashed', linewidth=2, label=median_label)

# Apply legend font size
ax.legend(fontsize=LEGEND_SIZE)


plt.tight_layout()
plt.savefig(hist_file_path, dpi=300) # dpi=300 for high quality
plt.close(fig) # Close the figure object
print(f"Histogram saved to: {hist_file_path}")

# --- PLOT 2: The Box Plot (Excellent for the thesis) ---
box_file_path = os.path.join(OUTPUT_DIR, 'global_latency_boxplot.png')

# Create figure and axes object
fig, ax = plt.subplots(figsize=(10, 4)) # Wider than it is tall

# Plot boxplot on the axes
ax.boxplot(final_latency_series.dropna(), vert=False) 

# --- STYLING ---
# ax.set_title(...) # <-- REMOVED
# ***** MODIFICATO *****
ax.set_xlabel('Latency [T_validation - T_0]', fontsize=LABEL_SIZE)

# ***** AGGIUNTO *****
# Imposta il limite massimo dell'asse X
ax.set_xlim(left=0, right=1500)

# Remove Y-axis ticks (not needed for a single box)
ax.set_yticks([])
ax.set_yticklabels([])

# Apply tick size only to X-axis
ax.tick_params(axis='x', labelsize=TICK_SIZE)

ax.grid(True, axis='x') # Add a grid only on the X-axis


plt.tight_layout()
plt.savefig(box_file_path, dpi=300)
plt.close(fig)
print(f"Box plot saved to: {box_file_path}")

print("\n--- Global processing complete. ---")

All aggregated results will be saved in: ../data\all\delay

Found 12 CSV files to aggregate...

--- Aggregation complete. ---
Total number of latencies calculated (from all subjects): 360

--- GLOBAL LATENCY STATISTICS (All Subjects) ---
Mean Latency: 700.68 ms
Std Deviation: 375.61 ms
Median (P50): 648.65 ms
P95 (95th Percentile): 1367.94 ms
Min: 31.00 ms | Max: 1943.71 ms

Metrics summary saved to: ../data\all\delay\summary_global_metrics.txt
Histogram saved to: ../data\all\delay\global_latency_histogram.png
Box plot saved to: ../data\all\delay\global_latency_boxplot.png

--- Global processing complete. ---
